![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20NB%20Header%20Banner.png)

## Exemplar Information

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Purpose:</b> This exemplar introduces students to molecular dynamics analysis using a real protein trajectory. It demonstrates how to load an educational dataset, visualise protein motion in 3D, calculate global stability metrics such as RMSD and radius of gyration, measure residue-level flexibility with RMSF, track domain movements over time, and interpret conformational change in a biologically meaningful way.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Intended audience / teaching context:</b> Designed for students and staff in introductory to intermediate computational biology, biochemistry, structural biology, or molecular dynamics teaching sessions. Suitable for classroom demonstration, guided lab work, workshops, or independent exploration of protein motion and conformational analysis.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Noteable Requirements:</b><br>
    <b>Environment:</b> LIVE 2026<br>
    <b>Server:</b> BioChemistry<br>
    <b>Kernel:</b> Python 3
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Required data or dependencies:</b>
    <br>
    - Precomputed molecular dynamics dataset for <b>adenylate kinase (adk_equilibrium)</b> via <code>MDAnalysisData</code><br>
    - Python packages: <code>numpy</code>, <code>pandas</code>, <code>plotly</code>, <code>ipywidgets</code>, <code>MDAnalysis</code>, <code>MDAnalysisData</code>, <code>nglview</code><br>
    - MDAnalysis modules: <code>MDAnalysis.analysis.align</code>, <code>MDAnalysis.analysis.rms</code><br>
    - Standard library modules: <code>warnings</code>
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Date created / last reviewed:</b> 13 July 2026<br>
    <b>Maintainer / owner:</b> Nik Yusuf
</div>


# Proteins in Motion: Exploring Molecular Dynamics and Conformational Change

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Why Protein Motion Matters

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Explore a real molecular dynamics trajectory and learn how motion can reveal stability, flexibility, and function.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    Proteins are not rigid sculptures. They breathe, twist, bend, and sometimes switch between functionally important shapes. A single crystal structure is a powerful snapshot, but a trajectory lets us watch how structure changes over time.
</div>

In this notebook, we will study <b>adenylate kinase (AdK)</b>, a classic enzyme used in molecular dynamics teaching. AdK helps cells manage energy by interconverting adenine nucleotides, and it is famous for having mobile regions that open and close during function.

Our central question is simple:

**What does the motion of this protein suggest about stability, flexibility, and domain behaviour?**

## 2. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Import the tools we need for loading trajectories, measuring structural change, drawing polished plots, and creating interactive controls.
</div>

Click the code cell below and press the <b><code>▶</code> Run</b> button in the toolbar at the top (or use <code>Shift + Enter</code>). The print statements will help you track our progress.

In [ ]:
print("Starting setup: importing molecular dynamics and visualisation libraries...")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets

import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
from MDAnalysisData import datasets
import nglview as nv


print("Success! Imports ready. We can now explore proteins in motion.")

## 3. Loading an Educational Molecular Dynamics Dataset

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Load a lightweight, reproducible molecular dynamics example from <b>MDAnalysisData</b> so everybody in the class starts from the same system.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    We will use a precomputed trajectory for <b>adenylate kinase</b>. This keeps the notebook practical for teaching: we do <b>not</b> run a simulation from scratch, but we still get to analyse real molecular motion.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try It:</b> The editable values in the next cell are safe to change. If you experiment with the smoothing window or the frame step, you can immediately see how that affects the visualisations.
</div>

In [ ]:
print("Loading a precomputed trajectory from MDAnalysisData...")

# Suppress the specific MDAnalysis DCD reader deprecation warning
warnings.filterwarnings(
    action="ignore",
    category=DeprecationWarning,
    message=".*DCDReader currently makes independent timesteps.*"
)

# EDIT THIS VALUE: this notebook is designed around the classic adenylate kinase example.
dataset_name = "adk_equilibrium"

# EDIT THIS VALUE: use a gentle rolling-average window for cleaner plots.
smoothing_window = 15

# EDIT THIS VALUE: increase this if you want a faster-moving frame slider later.
frame_step = 10

if dataset_name != "adk_equilibrium":
    raise ValueError("This flagship notebook is written around the adk_equilibrium teaching dataset.")

adk = datasets.fetch_adk_equilibrium()
u = mda.Universe(adk["topology"], adk["trajectory"])

protein = u.select_atoms("protein")
calphas = u.select_atoms("protein and name CA")

domain_selections = {
    "CORE": "protein and name CA and ((resid 1:29) or (resid 60:121) or (resid 160:214))",
    "NMP": "protein and name CA and resid 30:59",
    "LID": "protein and name CA and resid 122:159"
}

domain_groups = {name: u.select_atoms(selection) for name, selection in domain_selections.items()}

if any(group.n_atoms == 0 for group in domain_groups.values()):
    raise ValueError("One or more domain selections returned no atoms. Please check the residue numbering.")

u.trajectory[0]

print("Trajectory loaded successfully!")
print(f"Topology file: {adk['topology']}")
print(f"Trajectory file: {adk['trajectory']}")

## 4. First Look at the System

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Build a quick summary of the system before we start measuring motion. This helps us stay scientifically grounded: what exactly are we analysing?
</div>

Run the next cell to inspect the number of atoms, residues, frames, and the simple domain layout we will use in our analysis.

In [ ]:
print("Building a quick summary of the protein system...")

try:
    dt_ps = float(u.trajectory.dt)
except Exception:
    dt_ps = np.nan

if np.isfinite(dt_ps):
    total_time_ns = dt_ps * (u.trajectory.n_frames - 1) / 1000.0
    frame_time_label = f"{dt_ps:.3f} ps"
    total_time_label = f"{total_time_ns:.3f} ns"
else:
    total_time_label = "Time metadata not available"
    frame_time_label = "Time metadata not available"

summary_df = pd.DataFrame(
    {
        "Property": [
            "Protein system",
            "Trajectory example",
            "Protein atoms",
            "Protein residues",
            "Cα atoms",
            "Number of frames",
            "Time per frame",
            "Approximate total trajectory time"
        ],
        "Value": [
            "Adenylate kinase (AdK)",
            dataset_name,
            protein.n_atoms,
            protein.n_residues,
            calphas.n_atoms,
            u.trajectory.n_frames,
            frame_time_label,
            total_time_label
        ]
    }
)

domain_table = pd.DataFrame(
    {
        "Domain": ["CORE", "NMP", "LID"],
        "Residue range used here": ["1–29, 60–121, 160–214", "30–59", "122–159"],
        "Why it matters": [
            "Relatively stable scaffold for comparison",
            "Mobile nucleotide-binding region",
            "Mobile lid-like region involved in closing motion"
        ]
    }
)

print("Protein summary complete.")
display(summary_df)
display(domain_table)

## 5. Watching the Protein Move in 3D

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Use an interactive 3D viewer so we can literally watch the trajectory, not just calculate abstract numbers.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    This is one of the most important ideas in computational biochemistry: trajectories are best understood by combining <b>visual inspection</b> with <b>quantitative measurements</b>.
</div>

Run the next cell, then press play in the viewer controls. You can rotate, zoom, and pause the structure while it moves.

In [ ]:
print("Preparing an interactive trajectory viewer...")

u.trajectory[0]
traj_view = nv.show_mdanalysis(protein)
traj_view.clear_representations()
traj_view.add_cartoon(selection="protein", color_scheme="residueindex")

try:
    traj_view.center()
except Exception:
    pass

display(traj_view)
print("Interactive viewer ready. Use the play button to watch adenylate kinase move through time.")

## 6. Measuring Global Change: RMSD and Compactness

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Quantify how much the structure shifts over time and whether the protein stays globally compact.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>RMSD</b> tells us how far the structure has moved away from a reference frame. <b>Radius of gyration</b> tells us how spread out the protein mass is. Together, they help us distinguish overall drift from major collapse or expansion.
</div>

In [ ]:
print("Calculating RMSD, radius of gyration, and simple domain distances...")

# Suppress the specific MDAnalysis DCD reader deprecation warning
warnings.filterwarnings(
    action="ignore",
    category=DeprecationWarning,
    message=".*DCDReader currently makes independent timesteps.*"
)

reference = mda.Universe(adk["topology"], adk["trajectory"])
rmsd_analysis = rms.RMSD(u, reference, select="protein and name CA")
rmsd_analysis.run()

rmsd_df = pd.DataFrame(
    rmsd_analysis.results.rmsd,
    columns=["Frame", "Time (ps)", "Cα RMSD (Å)"]
)
rmsd_df["Frame"] = rmsd_df["Frame"].astype(int)

geometry_rows = []
for ts in u.trajectory:
    core_center = domain_groups["CORE"].center_of_geometry()
    nmp_center = domain_groups["NMP"].center_of_geometry()
    lid_center = domain_groups["LID"].center_of_geometry()
    geometry_rows.append(
        {
            "Frame": int(ts.frame),
            "Radius of gyration (Å)": float(protein.radius_of_gyration()),
            "CORE–NMP distance (Å)": float(np.linalg.norm(core_center - nmp_center)),
            "CORE–LID distance (Å)": float(np.linalg.norm(core_center - lid_center)),
            "NMP–LID distance (Å)": float(np.linalg.norm(nmp_center - lid_center))
        }
    )

geometry_df = pd.DataFrame(geometry_rows)
analysis_df = rmsd_df.merge(geometry_df, on="Frame")
analysis_df["Time (ns)"] = analysis_df["Time (ps)"] / 1000.0

metric_columns = [
    "Cα RMSD (Å)",
    "Radius of gyration (Å)",
    "CORE–NMP distance (Å)",
    "CORE–LID distance (Å)",
    "NMP–LID distance (Å)"
]

for column in metric_columns:
    analysis_df[f"{column} (smoothed)"] = analysis_df[column].rolling(
        window=smoothing_window,
        center=True,
        min_periods=1
    ).mean()

u.trajectory[0]

print("Trajectory metrics calculated successfully.")
display(analysis_df.head().round(3))

In [ ]:
print("Creating polished time-series views for global structural behaviour...")

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Cα RMSD over time", "Radius of gyration over time")
)

fig.add_trace(
    go.Scatter(
        x=analysis_df["Time (ns)"],
        y=analysis_df["Cα RMSD (Å)"],
        mode="lines",
        line=dict(color="rgba(76,114,176,0.25)", width=1),
        name="RMSD (raw)",
        showlegend=True
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        x=analysis_df["Time (ns)"],
        y=analysis_df["Cα RMSD (Å) (smoothed)"],
        mode="lines",
        line=dict(color="#4C72B0", width=3),
        name="RMSD (smoothed)",
        showlegend=True
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=analysis_df["Time (ns)"],
        y=analysis_df["Radius of gyration (Å)"],
        mode="lines",
        line=dict(color="rgba(85,168,104,0.25)", width=1),
        name="Rg (raw)",
        showlegend=True
    ),
    row=1,
    col=2
)
fig.add_trace(
    go.Scatter(
        x=analysis_df["Time (ns)"],
        y=analysis_df["Radius of gyration (Å) (smoothed)"],
        mode="lines",
        line=dict(color="#55A868", width=3),
        name="Rg (smoothed)",
        showlegend=True
    ),
    row=1,
    col=2
)

fig.update_xaxes(title_text="Time (ns)", row=1, col=1)
fig.update_xaxes(title_text="Time (ns)", row=1, col=2)
fig.update_yaxes(title_text="RMSD (Å)", row=1, col=1)
fig.update_yaxes(title_text="Radius of gyration (Å)", row=1, col=2)
fig.update_layout(
    template="plotly_white",
    height=450,
    title="Global stability and compactness in adenylate kinase"
)
fig.show()

global_summary = pd.DataFrame(
    {
        "Metric": ["Mean Cα RMSD", "Maximum Cα RMSD", "Mean radius of gyration", "Radius of gyration range"],
        "Value": [
            analysis_df["Cα RMSD (Å)"].mean(),
            analysis_df["Cα RMSD (Å)"].max(),
            analysis_df["Radius of gyration (Å)"].mean(),
            analysis_df["Radius of gyration (Å)"].max() - analysis_df["Radius of gyration (Å)"].min()
        ]
    }
)

display(global_summary.round(3))
print("Visualisation ready. Look for whether RMSD plateaus and whether compactness stays relatively stable.")

## 7. Finding Flexible Regions with RMSF

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Identify which residues move the most over the trajectory.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>RMSF</b> measures fluctuation residue-by-residue. Before calculating it, we align the trajectory so we do not confuse whole-protein tumbling with local flexibility.
</div>

In [ ]:
print("Aligning the trajectory and calculating per-residue RMSF...")

# Suppress the specific MDAnalysis DCD reader deprecation warning
warnings.filterwarnings(
    action="ignore",
    category=DeprecationWarning,
    message=".*DCDReader currently makes independent timesteps.*"
)

u_aligned = mda.Universe(adk["topology"], adk["trajectory"])
reference_aligned = mda.Universe(adk["topology"], adk["trajectory"])

align.AlignTraj(
    u_aligned,
    reference_aligned,
    select="protein and name CA",
    in_memory=True
).run()

calphas_aligned = u_aligned.select_atoms("protein and name CA")
rmsf_analysis = rms.RMSF(calphas_aligned).run()

rmsf_df = pd.DataFrame(
    {
        "Residue": calphas_aligned.resids,
        "Residue name": calphas_aligned.resnames,
        "RMSF (Å)": rmsf_analysis.results.rmsf
    }
)

def label_domain(resid):
    if 30 <= resid <= 59:
        return "NMP"
    if 122 <= resid <= 159:
        return "LID"
    return "CORE"

rmsf_df["Domain"] = rmsf_df["Residue"].apply(label_domain)
rmsf_df["Label"] = rmsf_df["Residue name"] + " " + rmsf_df["Residue"].astype(str)
top_flexible = rmsf_df.sort_values("RMSF (Å)", ascending=False).head(10).reset_index(drop=True)

print("RMSF calculated successfully.")
display(top_flexible.round(3))

In [ ]:
print("Visualising residue-level flexibility...")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=rmsf_df["Residue"],
        y=rmsf_df["RMSF (Å)"],
        mode="lines",
        line=dict(color="#2C7FB8", width=3),
        hovertemplate="Residue %{x}<br>RMSF %{y:.2f} Å<extra></extra>"
    )
)

fig.add_vrect(
    x0=30,
    x1=59,
    fillcolor="orange",
    opacity=0.14,
    line_width=0,
    annotation_text="NMP domain",
    annotation_position="top left"
)
fig.add_vrect(
    x0=122,
    x1=159,
    fillcolor="green",
    opacity=0.14,
    line_width=0,
    annotation_text="LID domain",
    annotation_position="top left"
)

fig.update_layout(
    template="plotly_white",
    title="Residue flexibility (RMSF) across adenylate kinase",
    xaxis_title="Residue number",
    yaxis_title="RMSF (Å)",
    height=450
)
fig.show()

display(top_flexible[["Label", "Domain", "RMSF (Å)"]].round(3))
print("Higher RMSF values often point to flexible loops or mobile domains rather than rigid structural core regions.")

## 8. Tracking Domain Motion Through Time

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Connect motion to biochemistry by tracking how the major domains move relative to one another.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    Adenylate kinase is famous for domain motion. If the distances between the CORE, NMP, and LID regions change over time, that is a simple structural proxy for opening and closing behaviour.
</div>

In [ ]:
print("Plotting domain-to-domain distances across the trajectory...")

distance_columns = [
    "CORE–NMP distance (Å)",
    "CORE–LID distance (Å)",
    "NMP–LID distance (Å)"
]

distance_colors = {
    "CORE–NMP distance (Å)": "#DD8452",
    "CORE–LID distance (Å)": "#55A868",
    "NMP–LID distance (Å)": "#C44E52"
}

fig = go.Figure()
for column in distance_columns:
    fig.add_trace(
        go.Scatter(
            x=analysis_df["Time (ns)"],
            y=analysis_df[column],
            mode="lines",
            line=dict(color=distance_colors[column], width=1),
            opacity=0.25,
            name=f"{column} (raw)",
            showlegend=False
        )
    )
    fig.add_trace(
        go.Scatter(
            x=analysis_df["Time (ns)"],
            y=analysis_df[f"{column} (smoothed)"],
            mode="lines",
            line=dict(color=distance_colors[column], width=3),
            name=column,
            showlegend=True
        )
    )

fig.update_layout(
    template="plotly_white",
    title="Relative movement of adenylate kinase domains",
    xaxis_title="Time (ns)",
    yaxis_title="Distance between domain centres (Å)",
    height=480
)
fig.show()

distance_summary = analysis_df[distance_columns].agg(["min", "max", "mean"]).T.reset_index()
distance_summary.columns = ["Measurement", "Minimum", "Maximum", "Mean"]
display(distance_summary.round(3))
print("These distance traces help us see whether the protein mainly stays in one basin or samples noticeably different conformations.")

## 9. Interactive Frame Explorer

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Link the numbers back to the movie. Choose a frame and a metric, and inspect what is happening at that exact point in the trajectory.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try It:</b> Slide to a frame where the CORE–LID distance is high, then compare it with a frame where it is low. This is a great way to build intuition for what “opening” and “closing” mean structurally.
</div>

In [ ]:
print("Building the interactive frame explorer...")

analysis_by_frame = analysis_df.set_index("Frame")
metric_options = {
    "Cα RMSD": "Cα RMSD (Å)",
    "Radius of gyration": "Radius of gyration (Å)",
    "CORE–NMP distance": "CORE–NMP distance (Å)",
    "CORE–LID distance": "CORE–LID distance (Å)",
    "NMP–LID distance": "NMP–LID distance (Å)"
}

def explore_frame(frame=0, metric_label="CORE–LID distance"):
    metric = metric_options[metric_label]
    frame = int(frame)
    selected = analysis_by_frame.loc[frame]

    try:
        traj_view.frame = frame
    except Exception:
        pass

    frame_summary = pd.DataFrame(
        {
            "Property": [
                "Frame",
                "Time (ns)",
                "Cα RMSD (Å)",
                "Radius of gyration (Å)",
                "CORE–NMP distance (Å)",
                "CORE–LID distance (Å)",
                "NMP–LID distance (Å)"
            ],
            "Value": [
                frame,
                float(selected["Time (ns)"]),
                float(selected["Cα RMSD (Å)"]),
                float(selected["Radius of gyration (Å)"]),
                float(selected["CORE–NMP distance (Å)"]),
                float(selected["CORE–LID distance (Å)"]),
                float(selected["NMP–LID distance (Å)"])
            ]
        }
    )

    display(frame_summary.round(3))

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=analysis_df["Frame"],
            y=analysis_df[metric],
            mode="lines",
            line=dict(color="#4C72B0", width=2),
            name=metric
        )
    )
    fig.add_vline(x=frame, line_color="crimson", line_width=3)
    fig.update_layout(
        template="plotly_white",
        title=f"{metric} across the trajectory",
        xaxis_title="Frame",
        yaxis_title=metric,
        height=360,
        showlegend=False
    )
    fig.show()

widgets.interact(
    explore_frame,
    frame=widgets.IntSlider(
        value=0,
        min=0,
        max=int(analysis_df["Frame"].max()),
        step=frame_step,
        description="Frame:",
        continuous_update=False
    ),
    metric_label=widgets.Dropdown(
        options=list(metric_options.keys()),
        value="CORE–LID distance",
        description="Metric:"
    )
);

print("Interactive explorer ready. Move the slider and compare the selected frame with the 3D viewer above.")

## 10. What Do These Dynamics Suggest Biologically?

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    A careful interpretation might be:
    <ul style="margin-top:8px;">
        <li>The protein remains globally folded, because its radius of gyration does not show dramatic collapse or expansion.</li>
        <li>The RMSD trace tells us the structure explores conformations that differ from the starting frame, but not in a way that suggests complete unfolding.</li>
        <li>The RMSF profile highlights that mobility is not evenly distributed: some regions are much more flexible than others.</li>
        <li>The domain-distance traces support the idea that AdK samples relative domain motions, especially in the mobile NMP and LID regions that are linked to function.</li>
    </ul>
</div>

This is exactly the kind of story molecular dynamics can reveal: the protein keeps its overall identity, but still samples structurally meaningful motions.

## 11. Limitations and Responsible Interpretation

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important:</b> A trajectory is not the same thing as biological truth. Molecular dynamics gives us a model of motion under a chosen force field, sampling scheme, and starting structure.
</div>

A few limitations are worth keeping in mind:

- This is a precomputed teaching trajectory, not a complete survey of every possible AdK conformation.
- RMSD, RMSF, and domain-centre distances are useful summaries, but they compress a lot of structural detail into a few numbers.
- A flexible region is not automatically a functional switch, and a distance change is not automatically a catalytic event.
- Our domain definitions are simplified and chosen for teaching clarity.
- Strong mechanistic claims would need additional simulation repeats, experimental evidence, or comparison with known biochemical data.

That is good scientific practice: use trajectories to generate insight, but interpret them cautiously.

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have loaded a real molecular dynamics trajectory, inspected protein motion in 3D, calculated multiple structural metrics, and linked those results to a biologically meaningful conformational story.
</div>

You have seen that:

- proteins are dynamic rather than static
- global stability and local flexibility can coexist
- residue-level and domain-level metrics reveal different kinds of motion
- interactive trajectory viewing makes the numbers much easier to interpret

Good next steps would be to:

- compare open and closed AdK trajectories
- analyse a different protein with a known conformational transition
- add contact maps or clustering
- compare multiple simulations to think about reproducibility

Keep exploring — this is where structural biology starts to feel alive.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20Notebook%20Footer.png)